**A. Membaca dan Eksplorasi Awal**

Baca dataset dari HDFS, tampilkan printSchema(), jumlah baris (count()), dan 10 baris pertama (show(10)).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, desc

# Membuat Spark session
spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .getOrCreate()

26/09/17 10:01:05 WARN Utils: Your hostname, manggala-IdeaPad-Slim-3-14IAH8 resolves to a loopback address: 127.0.1.1; using 192.168.1.105 instead (on interface wlp2s0)
26/09/17 10:01:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 10:01:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/17 10:01:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Path sesuai dengan lokasi file di HDFS yang dibuat pada cell persiapan
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"

# Membaca data
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# Menampilkan schema, jumlah baris, dan 10 baris pertama
df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

**B. Menangani Data Kosong**

Kolom rating memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan df.na.fill() atau df.na.drop() (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [3]:
# 1. Menampilkan jumlah data kosong pada kolom rating
jumlah_kosong = df.filter(col("rating").isNull()).count()
print(f"Jumlah data kosong pada rating: {jumlah_kosong}")

# 2. Menangani data kosong (kita pilih fill dengan nilai rata-rata atau angka default, misal 0)
df_clean = df.na.fill({"rating": 0})

Jumlah data kosong pada rating: 204


Saya memilih menggunakan fill() daripada drop() karena tidak ingin kehilangan data baris transaksi secara keseluruhan. Jika menggunakan drop(), saya akan kehilangan informasi penting seperti total_pendapatan atau kategori pada transaksi tersebut hanya karena pembeli tidak memberikan rating.

**C. Transformasi Data**

Tambahkan kolom total_pendapatan (unit_terjual x harga_satuan), lalu tambahkan kolom tier_transaksi yang bernilai "Besar" jika total_pendapatan > 500000, atau "Kecil" jika sebaliknya 

In [4]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

**D. Analisis dengan GroupBy**

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

In [5]:
# 1. Kategori dengan total_pendapatan tertinggi
print("1. Kategori dengan total_pendapatan tertinggi:")
df_transformed.groupBy("kategori").sum("total_pendapatan").orderBy(desc("sum(total_pendapatan)")).show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:")
df_transformed.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().orderBy(desc("count")).show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("3. Rata-rata rating per metode pembayaran:")
df_transformed.groupBy("metode_pembayaran").avg("rating").show()

1. Kategori dengan total_pendapatan tertinggi:
+------------+---------------------+
|    kategori|sum(total_pendapatan)|
+------------+---------------------+
|Rumah Tangga|            138665000|
+------------+---------------------+
only showing top 1 row

2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

3. Rata-rata rating per metode pembayaran:
+-----------------+------------------+
|metode_pembayaran|       avg(rating)|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



**E. Menyimpan Hasil ke HDFS**

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [6]:
# Path tujuan penyimpanan di HDFS (berupa direktori)
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september"

# Menyimpan dataframe ke HDFS
df_transformed.write.csv(output_path, header=True, mode="overwrite")

# Verifikasi apakah direktori sudah terbuat
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september

Found 2 items
-rw-r--r--   3 manggala supergroup          0 2026-09-17 10:09 /user/mahasiswa/tugas4/hasil_transaksi_september/_SUCCESS
-rw-r--r--   3 manggala supergroup      97296 2026-09-17 10:09 /user/mahasiswa/tugas4/hasil_transaksi_september/part-00000-f871e5de-542f-4924-bdcd-31a7fc803704-c000.csv
